# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow python-dotenv

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

/home/samin/.local/lib/python3.10/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


## 1. Configure MLflow

Fill in your assigned MLflow credentials in `HW02/HW02_B/.env` (see `.env.example`).

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name. Do not commit `.env` (it is in `.gitignore`).

In [2]:
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://185.50.38.163:33014")
MLFLOW_USERNAME = os.getenv("MLFLOW_USERNAME", "")
MLFLOW_PASSWORD = os.getenv("MLFLOW_PASSWORD", "")
EXPERIMENT_NAME = os.getenv("EXPERIMENT_NAME", "")

if (
    not MLFLOW_USERNAME
    or not MLFLOW_PASSWORD
    or not EXPERIMENT_NAME
    or MLFLOW_USERNAME == "student_your_username"
    or MLFLOW_PASSWORD == "your_mlflow_password"
):
    raise ValueError(
        "Set MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME in HW02/HW02_B/.env "
        "(copy from .env.example)."
    )

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

PROJECT_ROOT: /home/samin/bootcamps/MLOps-Course/HW02/HW02_B
MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_samin_kakaei
Experiment ID: 30


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [3]:
DATASET_VERSION = "v1_audit_cleaned"

FEATURE_DIR = PROJECT_ROOT / "data" / "features"

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# load the dataset (Parquet if it exists, otherwise CSV)
if parquet_path.exists():
    feature_df = pd.read_parquet(parquet_path)
elif csv_path.exists():
    feature_df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(f"Dataset not found. Expected {parquet_path} or {csv_path}.")

# load metadata if metadata_path exists
if metadata_path.exists():
    with open(metadata_path, encoding="utf-8") as f:
        metadata = json.load(f)
else:
    metadata = {}

print(feature_df.shape)
feature_df.head()

(10480, 33)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,room_type,property_type,accommodates,bedrooms,beds,...,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,total_reviews_before_cutoff,unique_reviewers_before_cutoff,avg_comment_len_before_cutoff,max_comment_len_before_cutoff,days_since_last_review,cutoff_date,dataset_version
0,1443670960781261954,30,30,1.0,0,Entire home/apt,Entire rental unit,2,1.0,1.0,...,1.0,2.0,30.0,5.0,5.0,371.200000,501.0,365.0,2026-08-11,v1_student
1,896043282611946316,30,0,0.0,1,Entire home/apt,Entire rental unit,2,1.0,NaN,...,0.0,5.0,25.0,2.0,2.0,469.000000,917.0,668.0,2026-08-11,v1_student
2,958726726744532841,30,0,0.0,1,Entire home/apt,Entire condo,2,1.0,NaN,...,0.0,2.0,7.0,23.0,23.0,277.304348,993.0,347.0,2026-08-11,v1_student
3,39969190,30,0,0.0,1,Entire home/apt,Entire rental unit,2,1.0,NaN,...,0.0,3.0,9.0,10.0,10.0,182.500000,423.0,546.0,2026-08-11,v1_student
4,1272264495001498383,30,30,1.0,0,Private room,Private room in townhouse,2,1.0,2.0,...,0.9,2.0,365.0,2.0,2.0,118.000000,201.0,557.0,2026-08-11,v1_student


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [4]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# check that TARGET_COL exists
if TARGET_COL not in feature_df.columns:
    raise ValueError(f"Target column {TARGET_COL!r} not found in dataset.")

# create y by selecting the target column from feature_df
y = feature_df[TARGET_COL]
clean_feature_cols = [col for col in feature_df.columns if col not in FORBIDDEN_MODEL_COLUMNS]
X_clean = feature_df[clean_feature_cols].copy()
for col in X_clean.columns:
    if X_clean[col].dtype == "boolean":
        X_clean[col] = X_clean[col].astype("float")

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())

print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: float64
Clean feature count: 26
['room_type', 'property_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost', 'neighbourhood_name', 'host_listing_count', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [5]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# create leaky_feature_cols by adding LEAKAGE_COLUMN to clean_feature_cols  
leaky_feature_cols = clean_feature_cols + [LEAKAGE_COLUMN]

# create X_leaky by selecting the leaky_feature_cols from feature_df
X_leaky = feature_df[leaky_feature_cols].copy()
for col in X_leaky.columns:
    if X_leaky[col].dtype == "boolean":
        X_leaky[col] = X_leaky[col].astype("float")

print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 27
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [6]:
# split X_clean and y 
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 26)
Test shape: (2096, 26)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [7]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# identify numeric_cols and categorical_cols from X_clean
numeric_cols = X_clean.select_dtypes(include=["int", "float"]).columns.tolist()
categorical_cols = [col for col in X_clean.columns if col not in numeric_cols]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 23
Categorical columns: 3


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [8]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)

# evaluate a fitted binary classifier
def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    y_score = get_positive_scores(model, X_test)
    y_pred = (y_score >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
    }

    return metrics, y_pred, y_score

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [9]:
ARTIFACT_DIR = PROJECT_ROOT / "outputs" / "mlflow_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# save run artifacts
def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    run_artifact_dir = ARTIFACT_DIR / run_name
    run_artifact_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots()
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax)
    fig.savefig(run_artifact_dir / "confusion_matrix.png", bbox_inches="tight")
    plt.close(fig)

    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    with open(run_artifact_dir / "classification_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    with open(run_artifact_dir / "feature_columns.json", "w", encoding="utf-8") as f:
        json.dump(feature_cols, f, indent=2)

    with open(run_artifact_dir / "dataset_metadata_snapshot.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return run_artifact_dir

## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [10]:
import tempfile

from mlflow.exceptions import MlflowException

# log sklearn pipeline model
def log_sklearn_pipeline_model(pipeline):
    """Log sklearn pipeline; fallback for older MLflow tracking servers."""
    try:
        mlflow.sklearn.log_model(pipeline, name="model")
    except MlflowException:
        with tempfile.TemporaryDirectory() as tmp_dir:
            model_path = Path(tmp_dir) / "model"
            mlflow.sklearn.save_model(pipeline, path=str(model_path))
            mlflow.log_artifacts(str(model_path), artifact_path="model")


# run mlflow experiment
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):

    with mlflow.start_run(run_name=run_name):
        pipeline.fit(X_train, y_train)

        # evaluate the pipeline
        metrics, y_pred, y_score = evaluate_binary_classifier(
            pipeline, X_test, y_test, threshold=threshold
        )

        artifact_dir = save_run_artifacts(
            run_name, y_test, y_pred, feature_cols, metadata
        )

        mlflow.log_params({**model_params, "threshold": threshold})
        mlflow.log_metrics(metrics)
        mlflow.set_tags(tags)
        mlflow.log_artifacts(str(artifact_dir))
        log_sklearn_pipeline_model(pipeline)

    return metrics

## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [11]:
X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

leaky_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols + [LEAKAGE_COLUMN]),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

leaky_pipeline = Pipeline(
    steps=[
        ("preprocessor", leaky_preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

run_mlflow_experiment(
    run_name="v0_leaky_logistic_regression",
    pipeline=leaky_pipeline,
    X_train=X_leaky_train,
    X_test=X_leaky_test,
    y_train=y_leaky_train,
    y_test=y_leaky_test,
    feature_cols=leaky_feature_cols,
    model_params={"max_iter": 1000, "random_state": RANDOM_STATE},
    tags={
        "leakage_status": "leaky",
        "known_defect": "uses future_available_rate_30d",
        "model_family": "logistic_regression",
    },
)

2026/06/05 13:00:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:00:37 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpwjng3s7_/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/30/runs/ba2fb0a5072a4bd3850ec3e1d599a545
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30


{'accuracy': 0.9985687022900763,
 'precision': 0.9993742177722152,
 'recall': 0.9987492182614134,
 'f1': 0.9990616202690021,
 'roc_auc': 0.9999911916778972}

## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [12]:
# build dummy pipeline
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

# run dummy baseline
run_mlflow_experiment(
    run_name="v1_dummy_baseline",
    pipeline=dummy_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"strategy": "most_frequent"},
    tags={"leakage_status": "clean", "model_family": "dummy"},
)

2026/06/05 13:01:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:01:04 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpwodprik0/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/30/runs/57618dcef4e640b8a311c9bb4e2f80b4
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30


{'accuracy': 0.762881679389313,
 'precision': 0.762881679389313,
 'recall': 1.0,
 'f1': 0.8654939106901218,
 'roc_auc': 0.5}

## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [13]:
# build clean logistic regression pipeline
clean_lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

# run clean logistic regression
run_mlflow_experiment(
    run_name="v2_clean_logistic_regression",
    pipeline=clean_lr_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"max_iter": 1000, "random_state": RANDOM_STATE},
    tags={"leakage_status": "clean", "model_family": "logistic_regression"},
)

2026/06/05 13:01:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:01:19 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpefs38ynh/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/30/runs/985b9dfc456641e39569b056e6cf741c
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30


{'accuracy': 0.9661259541984732,
 'precision': 0.9664224664224664,
 'recall': 0.9899937460913071,
 'f1': 0.978066110596231,
 'roc_auc': 0.9801460419804631}

## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [14]:
# build class-weighted logistic regression pipeline
balanced_lr_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
                class_weight="balanced",
            ),
        ),
    ]
)

# run class-weighted logistic regression
run_mlflow_experiment(
    run_name="v3_balanced_logistic_regression",
    pipeline=balanced_lr_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={
        "max_iter": 1000,
        "random_state": RANDOM_STATE,
        "class_weight": "balanced",
    },
    tags={"leakage_status": "clean", "model_family": "logistic_regression"},
)

2026/06/05 13:01:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:01:32 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp2jhf6ilu/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/30/runs/60a634490d874c7da5839933a4d007e8
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30


{'accuracy': 0.9723282442748091,
 'precision': 0.9753238741517581,
 'recall': 0.9887429643527205,
 'f1': 0.9819875776397515,
 'roc_auc': 0.9813779487431153}

## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [15]:
THRESHOLDS = [0.30, 0.40, 0.50, 0.60]

# build threshold pipeline
threshold_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

# run threshold tuning
for threshold in THRESHOLDS:
    metrics = run_mlflow_experiment(
        run_name=f"v4_threshold_{threshold:.2f}",
        pipeline=threshold_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={"max_iter": 1000, "random_state": RANDOM_STATE},
        tags={"leakage_status": "clean", "model_family": "logistic_regression"},
        threshold=threshold,
    )
    print(threshold, metrics)

2026/06/05 13:01:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:01:45 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp61ubghdw/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v4_threshold_0.30 at: http://185.50.38.163:33014/#/experiments/30/runs/3bfc7df6410947d6adcfd511a82979f2
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30
0.3 {'accuracy': 0.9637404580152672, 'precision': 0.963481436396835, 'recall': 0.9899937460913071, 'f1': 0.9765576804441702, 'roc_auc': 0.9801460419804631}


2026/06/05 13:01:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:01:50 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp2vss9p_i/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v4_threshold_0.40 at: http://185.50.38.163:33014/#/experiments/30/runs/db48be8685c34ec1beedecaa60fc685a
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30
0.4 {'accuracy': 0.9642175572519084, 'precision': 0.964068209500609, 'recall': 0.9899937460913071, 'f1': 0.9768589941376119, 'roc_auc': 0.9801460419804631}


2026/06/05 13:01:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:01:55 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpb1juibwh/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v4_threshold_0.50 at: http://185.50.38.163:33014/#/experiments/30/runs/2d1ad29bc70b4350ae3ff4c4a2c38859
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30
0.5 {'accuracy': 0.9661259541984732, 'precision': 0.9664224664224664, 'recall': 0.9899937460913071, 'f1': 0.978066110596231, 'roc_auc': 0.9801460419804631}


2026/06/05 13:01:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:02:01 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp043hhzwb/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v4_threshold_0.60 at: http://185.50.38.163:33014/#/experiments/30/runs/73b7c97daad44c799a4b26f5272d45d7
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30
0.6 {'accuracy': 0.9694656488549618, 'precision': 0.970570202329859, 'recall': 0.9899937460913071, 'f1': 0.9801857585139319, 'roc_auc': 0.9801460419804631}


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [16]:
# build tree-based pipeline
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                max_depth=10,
                min_samples_leaf=5,
                class_weight=None,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

# run tree-based model  
run_mlflow_experiment(
    run_name="v5_random_forest",
    pipeline=rf_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 5,
        "class_weight": "none",
        "random_state": RANDOM_STATE,
    },
    tags={"leakage_status": "clean", "model_family": "random_forest"},
)

2026/06/05 13:02:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/06/05 13:02:22 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpf2w9fk0w/model/model.pkl, flavor: sklearn). Fall back to return ['scikit-learn==1.7.2', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 


🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/30/runs/a37a223cfd294ac2a27516b90d5a795c
🧪 View experiment at: http://185.50.38.163:33014/#/experiments/30


{'accuracy': 0.9790076335877863,
 'precision': 0.9820210787352759,
 'recall': 0.9906191369606003,
 'f1': 0.9863013698630136,
 'roc_auc': 0.9871952163260992}

## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [17]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# retrieve runs
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
)

# build comparison dataframe
comparison_df = runs_df[
    [
        "run_id",
        "tags.mlflow.runName",
        "tags.leakage_status",
        "tags.model_family",
        "metrics.accuracy",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.roc_auc",
    ]
].rename(
    columns={
        "tags.mlflow.runName": "run_name",
        "tags.leakage_status": "leakage_status",
        "tags.model_family": "model_family",
        "metrics.accuracy": "accuracy",
        "metrics.precision": "precision",
        "metrics.recall": "recall",
        "metrics.f1": "f1",
        "metrics.roc_auc": "roc_auc",
    }
)

comparison_df

,run_id,run_name,leakage_status,model_family,accuracy,precision,recall,f1,roc_auc
0,a37a223cfd294ac2a27516b90d5a795c,v5_random_forest,clean,random_forest,0.979008,0.982021,0.990619,0.986301,0.987195
1,73b7c97daad44c799a4b26f5272d45d7,v4_threshold_0.60,clean,logistic_regression,0.969466,0.970570,0.989994,0.980186,0.980146
2,2d1ad29bc70b4350ae3ff4c4a2c38859,v4_threshold_0.50,clean,logistic_regression,0.966126,0.966422,0.989994,0.978066,0.980146
3,db48be8685c34ec1beedecaa60fc685a,v4_threshold_0.40,clean,logistic_regression,0.964218,0.964068,0.989994,0.976859,0.980146
4,3bfc7df6410947d6adcfd511a82979f2,v4_threshold_0.30,clean,logistic_regression,0.963740,0.963481,0.989994,0.976558,0.980146
5,60a634490d874c7da5839933a4d007e8,v3_balanced_logistic_regression,clean,logistic_regression,0.972328,0.975324,0.988743,0.981988,0.981378
6,985b9dfc456641e39569b056e6cf741c,v2_clean_logistic_regression,clean,logistic_regression,0.966126,0.966422,0.989994,0.978066,0.980146
7,57618dcef4e640b8a311c9bb4e2f80b4,v1_dummy_baseline,clean,dummy,0.762882,0.762882,1.000000,0.865494,0.500000
8,ba2fb0a5072a4bd3850ec3e1d599a545,v0_leaky_logistic_regression,leaky,logistic_regression,0.998569,0.999374,0.998749,0.999062,0.999991
9,77a0080f79964255a7b0a208960a8f0f,v4_threshold_0.60,clean,logistic_regression,0.969466,0.970570,0.989994,0.980186,0.980146


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [18]:
# select best clean run
candidate_runs = comparison_df[
    (comparison_df["leakage_status"] == "clean")
    & comparison_df["f1"].notna()
    & (comparison_df["model_family"] != "dummy")
    & (~comparison_df["run_name"].str.startswith("v4_threshold"))
].drop_duplicates(subset=["run_name"], keep="first")

candidate_runs = candidate_runs.sort_values(["f1", "roc_auc"], ascending=False)

print("Top clean model candidates:")
print(
    candidate_runs[
        ["run_name", "model_family", "precision", "recall", "f1", "roc_auc"]
    ].head(5)
)

best_run = candidate_runs.iloc[0]
BEST_RUN_ID = best_run["run_id"]

if BEST_RUN_ID is None:
    raise ValueError("Set BEST_RUN_ID to your selected clean MLflow run ID.")

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("\nSelected best run:", BEST_RUN_ID)
print("Selected run name:", best_run["run_name"])
print(
    "Selected metrics:",
    {
        "accuracy": best_run["accuracy"],
        "precision": best_run["precision"],
        "recall": best_run["recall"],
        "f1": best_run["f1"],
        "roc_auc": best_run["roc_auc"],
    },
)

Selected best run: a37a223cfd294ac2a27516b90d5a795c
Selected run name: v5_random_forest


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [19]:
final_explanation = """
I selected v5_random_forest (run a37a223cfd294ac2a27516b90d5a795c) as the final production candidate.
Among all clean model runs, it had the best test F1 (0.986) and ROC-AUC (0.987), while keeping strong precision (0.982) and recall (0.991).
It also logs a full sklearn Pipeline with preprocessing, which makes the model reproducible and ready for serving.

I rejected v0_leaky_logistic_regression even though its metrics looked best (F1 ≈ 0.999).
That run used future_available_rate_30d, which comes from the label window and is too close to the target definition.
It would not work in real production because that future information is not available at prediction time.

Compared to the dummy baseline (accuracy ≈ 0.763), all clean real models added meaningful value.
The balanced logistic model improved slightly over the plain logistic model, and threshold tuning showed the expected precision/recall tradeoff at fixed ROC-AUC.

Next, I would tune RandomForest hyperparameters, compare against other tree-based models, and run cross-validation to confirm the result is stable.
"""

print(final_explanation)


I selected v5_random_forest (run a37a223cfd294ac2a27516b90d5a795c) as the final production candidate.
Among all clean model runs, it had the best test F1 (0.986) and ROC-AUC (0.987), while keeping strong precision (0.982) and recall (0.991).
It also logs a full sklearn Pipeline with preprocessing, which makes the model reproducible and ready for serving.

I rejected v0_leaky_logistic_regression even though its metrics looked best (F1 ≈ 0.999).
That run used future_available_rate_30d, which comes from the label window and is too close to the target definition.
It would not work in real production because that future information is not available at prediction time.

Compared to the dummy baseline (accuracy ≈ 0.763), all clean real models added meaningful value.
The balanced logistic model improved slightly over the plain logistic model, and threshold tuning showed the expected precision/recall tradeoff at fixed ROC-AUC.

Next, I would tune RandomForest hyperparameters, compare against o